# BANKING77 integrated group project

Run from the repository root in the documented Python 3.12 environment. The first model call downloads the pinned Qwen checkpoint. This notebook uses the fixed evaluation sample, keeps reference labels out of prompts, exercises the validated JSON/tool pipeline, and writes a JSONL run record plus a metrics summary. Tools use mock data only.


In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import json, platform, uuid
import pandas as pd
import torch, transformers
from src.logging_config import configure_logging
from src.model.llm_baseline import load_model, generate_response, MODEL_ID, MODEL_REVISION
from src.structured.pipeline import handle_query
from src.safety.guardrails import check_output

configure_logging()
ROOT = Path.cwd()
if not (ROOT / 'data/evaluation_sample.csv').exists():
    raise RuntimeError('Run this notebook with the repository root as the working directory.')
DATA = pd.read_csv(ROOT / 'data/evaluation_sample.csv')
assert {'sample_id','text','intent'} <= set(DATA.columns)
ALLOWED_INTENTS = sorted(DATA['intent'].dropna().astype(str).unique())
assert len(ALLOWED_INTENTS) == 77
RUN_ID = 'integrated_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '_' + uuid.uuid4().hex[:8]
print({'run_id': RUN_ID, 'samples': len(DATA), 'model': MODEL_ID, 'revision': MODEL_REVISION})

## Load the model and run the integrated pipeline

For a quick smoke demonstration, set `LIMIT` to a small number. For the reported evaluation, use all 231 rows and state the actual denominator. The classifier is evaluated separately in Part 2; here the pipeline's structured `intent` is scored alongside schema validity, safety status, tool status, and a human-review placeholder for response quality.

In [ ]:
LIMIT = len(DATA)  # set to e.g. 5 only for a smoke/demo run
model, tokenizer = load_model()
def generate(prompt):
    return generate_response(model, tokenizer, prompt, {'do_sample': False, 'max_new_tokens': 200})

rows = []
for row in DATA.head(LIMIT).itertuples(index=False):
    result = handle_query(str(row.text), ALLOWED_INTENTS, generate)
    safety = check_output(result['final_response'])
    rows.append({
        'run_id': RUN_ID, 'sample_id': str(row.sample_id), 'query': str(row.text),
        'actual_intent': str(row.intent), 'predicted_intent': result['intent'],
        'intent_correct': result['intent'] == str(row.intent),
        'status': result['status'], 'json_generated': result['raw_output'] is not None,
        'json_valid': result['raw_output'] is not None and result['status'] != 'invalid_output',
        'output_safety_status': safety.status, 'tool': result['tool'],
        'tool_result': result['tool_result'], 'response': result['final_response'],
        'response_quality_human_score': None,
    })
results = pd.DataFrame(rows)
results.head()

## Demonstrate tool and failure paths

These synthetic examples exercise the same production pipeline with deterministic model text. They are demonstrations, not model-quality measurements, and are kept out of the evaluation metrics.

In [ ]:
demo_outputs = iter([
    '{"intent":"cash_withdrawal","response":"I will check the demo transaction.","action_required":true,"action":{"tool":"transaction_status","arguments":{"transaction_id":"TXN1002"}}}',
    'not JSON',
    '{"intent":"cash_withdrawal","response":"Please send me your PIN.","action_required":false,"action":null}',
]
demo_queries = ['What is the status of TXN1002?', 'Show malformed output handling', 'Check my request safely']
demo_results = [handle_query(q, ALLOWED_INTENTS, lambda _prompt: next(demo_outputs)) for q in demo_queries]
pd.DataFrame([{k: r[k] for k in ('query','status','tool','tool_result','final_response')} for r in demo_results])

## Metrics and review

Intent correctness and schema validity are automated. Safety is represented by pipeline outcomes and a second guardrail check, which does not prove semantic safety. Response quality needs human scoring; review the generated responses for relevance, clarity, intent consistency, and unsupported claims, then fill the score column using the team's documented rubric. Do not call a blank human score a quality result.

In [ ]:
generated = results[results['json_generated']]
metrics = {
    'run_id': RUN_ID, 'n': int(len(results)), 'generated_n': int(len(generated)),
    'intent_accuracy': float(results['intent_correct'].mean()) if len(results) else None,
    'json_schema_valid_rate': float(generated['json_valid'].mean()) if len(generated) else None,
    'response_quality_human_scored_n': int(results['response_quality_human_score'].notna().sum()),
    'safe_pipeline_status_rate': float((results['status'] == 'ok').mean()) if len(results) else None,
    'tool_call_n': int(results['tool'].notna().sum()),
    'model_id': MODEL_ID, 'model_revision': MODEL_REVISION,
    'python': platform.python_version(), 'torch': torch.__version__,
    'transformers': transformers.__version__, 'decoding': {'do_sample': False, 'max_new_tokens': 200},
    'sample_ids': results['sample_id'].tolist(),
}
OUT = ROOT / 'experiments'
OUT.mkdir(exist_ok=True)
results.to_csv(OUT / f'{RUN_ID}_outputs.csv', index=False)
(OUT / f'{RUN_ID}_metrics.json').write_text(json.dumps(metrics, indent=2), encoding='utf-8')
with (OUT / 'experiment_runs.jsonl').open('a', encoding='utf-8') as log:
    for record in results.to_dict(orient='records'):
        log.write(json.dumps(record, ensure_ascii=False, default=str) + '\n')
print(json.dumps(metrics, indent=2))